# 22 — Nesting

Last stage: the harmonised intra-US table of step 21 is inserted into the OECD ICIO frame,
the world block is carried over unchanged, and every flow between the world and the United
States is expanded over the 51 state regions. Output: `nested_mriot_<year>.parquet`, the
delivered files.

**All of the logic lives in `nest_v31.py`.** This notebook only drives it and checks the
result, so that the module is the single definition of the construction and the delivered
files cannot depend on which of the two was run.

Two allocators do the expansion, and they answer different questions:

* the **production share** $S_{s,j}$, state $s$'s share of United States gross product in
  sector $j$ (BEA SAGDP2), splits the *intermediate* flows between the world and the United
  States, and the state of *origin* of exports for final use;
* the **destination allocator** $\Theta^{c}_{s}$ splits the *destination* of final demand,
  category by category:

$$\Theta^{c}_{s'}=
\begin{cases}
\dfrac{\sum_j \Phi^{c}(s',j)}{\sum_{s''}\sum_j \Phi^{c}(s'',j)}
  & c \in \{\text{household consumption},\ \text{investment}\},\\[10pt]
\dfrac{G_{s'}}{\sum_{s''} G_{s''}} & c = \text{government final consumption},
\end{cases}$$

with $\Phi^{c}$ the sub-national final demand of category $c$ and $G$ gross state product.
Both sum to one over states, so every expansion conserves the corresponding total of the
global table exactly. The comparison that settles the choice of $\Theta$ — against personal
consumption expenditure by state and four government benchmarks — is in
`validation/final_demand_allocator.ipynb`.

In [ ]:
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))          # so `import nest_v31` resolves

import numpy as np, pandas as pd
from paths import ROOT
import nest_v31 as N

print(f"WiNDC build   : {N.WINDC_VER}")
print(f"VA source     : {N.VA_SOURCE}")
print(f"FD allocator  : {N.F_SOURCE}")
print(f"harmonised in : {N.WINDC_HARM.relative_to(ROOT)}")
print(f"output to     : {N.OUT_ROOT.relative_to(ROOT)}")

## The two share tables

`build_shares_pivot` gives the sectoral production shares that make up $S$;
`build_gdp_share_pivot` gives the all-industry state product share, which is the government
column of $\Theta$. Both are read once and reused for every year.

In [ ]:
shares_pivot = N.build_shares_pivot()
gdp_pivot    = N.build_gdp_share_pivot()
print(f"sectoral shares : {len(shares_pivot):,} (state, sector, year) rows")
print(f"product shares  : {len(gdp_pivot):,} (state, year) rows")

# the government allocator, largest states, reference year
g = gdp_pivot.loc[(slice(None), 2017)].sort_values(ascending=False)
print("\nstate product share 2017, ten largest, %:")
print((g.head(10) * 100).round(2).to_string())

## One year, with the conservation checks

Nesting is exact by construction: no fitting step and no tolerance enter the expansion of
the world-to-United-States accounts. The checks below are therefore equalities, not
approximations, and a failure means a coding error rather than a convergence problem.

In [ ]:
YEAR = 2017
nested = N.nest_year(YEAR, shares_pivot, gdp_pivot, N.VA_SOURCE, N.F_SOURCE)
print(f"nested {YEAR}: {nested.shape[0]:,} x {nested.shape[1]:,}")

# ── conservation: the 51 state columns must sum back to the OECD USA aggregate ──
df_o  = pd.read_parquet(N.find_oecd_file(YEAR))
states = sorted(set(l.split("_")[0] for l in
                    np.load(N.WINDC_HARM / f"IOT_{YEAR}_harmonized.npz",
                            allow_pickle=True)["index_labels"]))
secs   = [r.split("_", 1)[1] for r in df_o.index if r.startswith("USA_")]
world  = [r for r in df_o.index if "_" in r and len(r.split("_")[0]) == 3
          and not r.startswith("USA_") and r.split("_", 1)[1] in secs]

def rel(a, b):  return abs(a - b) / max(abs(b), 1.0)

checks = []
for cat in N.FD_CATS:                              # imported final demand, per category
    got = nested.loc[world, [f"{s}_{cat}" for s in states]].values.sum()
    ref = df_o.loc[world, f"USA_{cat}"].values.sum()
    checks.append((f"world -> states, {cat}", got, ref))
for cat in N.FD_CATS:                              # intra-US final demand, per category
    got = nested.loc[[f"{s}_{j}" for s in states for j in secs],
                     [f"{s}_{cat}" for s in states]].values.sum()
    ref = df_o.loc[[f"USA_{j}" for j in secs], f"USA_{cat}"].values.sum()
    checks.append((f"states -> states, {cat}", got, ref))
got = nested.loc[world, [f"{s}_{j}" for s in states for j in secs]].values.sum()
ref = df_o.loc[world, [f"USA_{j}" for j in secs]].values.sum()
checks.append(("world -> states, intermediate", got, ref))

CH = pd.DataFrame(checks, columns=["block", "nested (M$)", "OECD (M$)"])
CH["rel. error"] = [rel(a, b) for _, a, b in checks]
print(CH.to_string(index=False, float_format=lambda v: f"{v:,.4g}"))
assert CH["rel. error"].max() < 1e-9, "conservation broken"
print(f"\nall blocks conserved, worst relative error {CH['rel. error'].max():.1e}")

## The delivered series, 1997–2022

The full loop. 1997 is the first year of the sub-national source and 2022 the last of the
global table; 2023 exists sub-nationally but has no counterpart in the global table and is
therefore not delivered.

In [ ]:
YEARS = list(range(1997, 2023))

for year in YEARS:
    out = N.OUT_ROOT / f"nested_mriot_{year}.parquet"
    print(f"[{year}]", end=" ", flush=True)
    tbl = N.nest_year(year, shares_pivot, gdp_pivot, N.VA_SOURCE, N.F_SOURCE)
    tbl.to_parquet(out, engine="fastparquet", compression="gzip")
    print(f"{tbl.shape[0]:,}x{tbl.shape[1]:,} -> {out.name}", flush=True)
    del tbl

print(f"\n{len(YEARS)} files written to {N.OUT_ROOT.relative_to(ROOT)}")

The accounting checks on the delivered files — world and state row and column balance,
symmetry of the output row and column, bit-level invariance of the non-US block — are run
by `validation/check_nested_mriot.ipynb`.